In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()))

from agent_res import build_agent, interviewBuilder

agent = build_agent()
builder = interviewBuilder()

graph = agent.get_graph()
builder_graph = builder.get_graph()

print(f"Agent compiled. Nodes: {list(agent.nodes.keys())}")
print(f"Builder compiled. Nodes: {list(builder_graph.nodes.keys())}")

In [ ]:
from IPython.display import Markdown

bulder_mermaid_code = builder_graph.draw_mermaid()
display(Markdown(f"```mermaid\n{bulder_mermaid_code}\n```"))

mermaid_code = graph.draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))


In [ ]:
from IPython.display import Image

builder_png_data = builder_graph.draw_mermaid_png()
display(Image(builder_png_data))

png_data = graph.draw_mermaid_png()
display(Image(png_data))

In [ ]:
# Inputs
max_analysts = 3 
topic = "The benefits of adopting LangGraph as an agent framework"
thread = {"configurable": {"thread_id": "1"}}

# Run the graph until the first interruption
for event in agent.stream(
    {
        "topic":topic, 
        "max_analysts":max_analysts
    }, 
    thread, 
    stream_mode="values"
):
    
    analysts = event.get('analysts', '')
    if analysts:
        for analyst in analysts:
            print(f"Name: {analyst.name}")
            print(f"Affiliation: {analyst.affiliation}")
            print(f"Role: {analyst.role}")
            print(f"Description: {analyst.description}")
            print("-" * 50)  

In [ ]:
# Get state and look at next node
state = agent.get_state(thread)
state.next

In [ ]:
# We now update the state as if we are the human_feedback node
agent.update_state(thread, {"human_analyst_feedback": 
                            "Add in someone from a startup to add an entrepreneur perspective"}, as_node="human_feedback")

In [ ]:
# Continue the graph execution
for event in agent.stream(None, thread, stream_mode="values"):
    # Review
    analysts = event.get('analysts', '')
    if analysts:
        for analyst in analysts:
            print(f"Name: {analyst.name}")
            print(f"Affiliation: {analyst.affiliation}")
            print(f"Role: {analyst.role}")
            print(f"Description: {analyst.description}")
            print("-" * 50) 

In [ ]:
# If we are satisfied, then we simply supply no feedback
further_feedack = 'approve'
agent.update_state(thread, {"human_analyst_feedback": further_feedack}, as_node="human_feedback")

In [ ]:
# Continue
for event in agent.stream(None, thread, stream_mode="updates"):
    print("--Node--")
    node_name = next(iter(event.keys()))
    print(node_name)

In [ ]:
from IPython.display import Markdown
final_state = agent.get_state(thread)
report = final_state.values.get('final_report')
print(report)